# PathSnap end-to-end workflow

Run this notebook from a clone of PathSnap. It uses the same Python modules as the command-line workflow, so preprocessing, features, and evaluation agree with the repository.

Install dependencies in the notebook environment with `python -m pip install -r requirements-training.txt jupyterlab`, then start JupyterLab from the repo. Set your dataset path below. Enable the stages you want to run. With the switches off, the notebook reads any existing reports and artifacts.

In [ ]:
import json
import subprocess
import sys
from pathlib import Path

root = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / 'src' / 'training' / 'train.py').is_file()), None)
if root is None:
    raise RuntimeError('Open this notebook from inside a PathSnap clone.')
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

DATA_ROOT = Path(r'C:\path\to\Student folder')
RUN_PREPARATION = False
RUN_TRAINING = False
RUN_ALIGNMENT_AUDIT = False
RUN_BAYESIAN_SEARCH = False
RUN_ROUTE_EVALUATION = False
MODELS_TO_TRAIN = ['logistic']  # Other choices: knn, forest, boosting, mlp, tree (if available).

def run(*args):
    subprocess.run([sys.executable, *map(str, args)], cwd=root, check=True)

def show_json(relative_path):
    path = root / relative_path
    if path.is_file():
        print(json.dumps(json.loads(path.read_text(encoding='utf-8')), indent=2))
    else:
        print(f'No report yet: {relative_path}')


## 1. Preprocess the GPS and road-network files

Read track, node, arc and reference-route files. Validate identifiers and coordinates, preserve trip splits, and project coordinates into local metres. The reference routes stay separate from model inputs.

In [ ]:
if RUN_PREPARATION:
    if not all((DATA_ROOT / split).is_dir() for split in ('train', 'student_test')):
        raise FileNotFoundError('Set DATA_ROOT to the directory containing train and student_test.')
    run('src/data_processing/preprocess.py', '--data-root', DATA_ROOT.resolve())
show_json('results/preprocessing/summary.json')


## 2. Generate candidate roads and features

The default search selects up to eight directed arcs within 100 m and uses the nearest arcs as a flagged fallback. Each candidate has geometric, heading, timing, and connectivity features.

In [ ]:
if RUN_PREPARATION:
    run('src/features/candidates.py')
show_json('results/features/summary.json')

import numpy as np

candidate_file = next(iter(sorted((root / 'data/features/student_test').glob('*/candidates.npz'))), None)
if candidate_file is not None:
    with np.load(candidate_file, allow_pickle=False) as candidates:
        names = candidates['feature_names'].tolist()
        print('Trip:', candidate_file.parent.name)
        print('Feature matrix shape:', candidates['features'].shape)
        print('First GPS observation candidates:', candidates['candidate_arc_ids'][:candidates['offsets'][1]])
        print('First candidate features:', dict(zip(names, candidates['features'][0].tolist())))


## 3. Align reference routes and audit candidate coverage

The aligner derives approximate per-observation labels from the ordered route. It excludes points more than 100 m from the assigned reference arc. The optional audit checks all observations on smaller trips and samples larger trips, with all fallbacks checked.

In [ ]:
if RUN_PREPARATION:
    run('src/training/align_routes.py')
if RUN_ALIGNMENT_AUDIT:
    run('src/features/audit.py', '--full-max-observations', '500')
show_json('results/feature_audit/summary.json')


## 4. Train and compare models

The runner uses three trip-grouped validation folds from the 80 training trips. It refits each selected model on those trips and scores the 10 student-test trips. Top-1 measures how often the highest-scoring candidate matches the aligned reference road.

In [ ]:
if RUN_TRAINING:
    run('src/training/train.py', '--models', *MODELS_TO_TRAIN)

print(f"{'Model':<18} {'CV top-1':>10} {'Test top-1':>11} {'Test macro F1':>14}")
for path in sorted((root / 'results/training').glob('*.json')):
    report = json.loads(path.read_text(encoding='utf-8'))
    test = report.get('student_test', report)
    if 'top1_accuracy' not in test:
        continue
    cv = report.get('cv_top1')
    cv_text = f'{cv:.2%}' if cv is not None else '—'
    print(f"{report.get('model', path.stem):<18} {cv_text:>10} "
          f"{test['top1_accuracy']:>11.2%} {test['macro_f1']:>14.2%}")


## 5. Optional Bayesian tuning for Histogram Gradient Boosting

This stage needs `requirements-optimization.txt` and the HGB-only tuning code from the `bayesian-optimization` branch. Enable the switch when you intend to run the 30 trials. The student test split is scored after selection using the training-trip folds.

In [ ]:
if RUN_BAYESIAN_SEARCH:
    if not (root / 'src/optimization/tune.py').is_file():
        raise FileNotFoundError('Add the Histogram Gradient Boosting tuning code from bayesian-optimization.')
    run('-m', 'src.optimization.tune', '--trials', '30')
optimization_report = root / 'results/optimization/boosting_bayesian.json'
if optimization_report.is_file():
    report = json.loads(optimization_report.read_text(encoding='utf-8'))
    for field in ('objective', 'best_params', 'best_cv', 'student_test'):
        print(field, report[field])
else:
    print('No Histogram Gradient Boosting tuning report yet.')


## 6. Reconstruct a trip and launch the viewer

The current demo uses the logistic model and Viterbi decoder. It joins plausible road states across successive observations and starts new pieces when the time gap is large or no transition is feasible.

From a separate terminal at the repository root, run `python -m api.server` and open `http://127.0.0.1:8000`. See `deployment/README.md` for the two-laptop demo.

In [ ]:
if RUN_ROUTE_EVALUATION:
    run('-m', 'src.route_reconstruction.evaluate')
show_json('results/metrics/logistic_decoder.json')
